# VAE: Auto-Encoding Variational Bayes (Kingma & Welling, ICLR 2014)
### Triển Khai Bằng PyTorch & Thử Nghiệm Đối Chiếu Giá Trị Học Thuật

> **Tác giả:** Diederik P. Kingma, Max Welling  
> **Hội nghị:** ICLR 2014 (Banff, Canada)  
> **Ý nghĩa lịch sử:** Kết hợp học sâu và suy diễn biến phân Bayes. Biến Autoencoder thành mô hình tạo sinh (Generative Model) có khả năng sinh mẫu mới và nội suy mượt mà trong không gian ẩn.

---

## 🎯 Mục Tiêu Bài Thực Hành
1. **Kiến trúc VAE:** Xây dựng mạng Encoder (dự đoán $\mu$ và $\log\sigma^2$), bộ tái tham số hóa `reparameterize`, và mạng Decoder.
2. **Hàm mất mát ELBO:** Cân bằng giữa sai số tái tạo BCE và độ phân kỳ KL Divergence.
3. **Bản đồ đa tạp 2D:** Quét lưới tọa độ không gian ẩn $[-2.5, 2.5] \times [-2.5, 2.5]$ và giải mã để quan sát sự chuyển tiếp hình học liên tục.
4. **Academic Value Benchmark:** Đối chiếu định lượng 3 trường phái:
   - *Mô hình 1 (Deterministic Autoencoder - Hinton 2006):* Không gian ẩn rỗng, lấy mẫu ngẫu nhiên sinh ảnh rác.
   - *Mô hình 2 (VAE Không KL / $\beta=0$):* Tái tạo tốt nhưng không gian ẩn phân tán tự do.
   - *Mô hình 3 (Full Variational Autoencoder - Kingma 2013):* Cân bằng hoàn hảo ELBO, sinh mẫu sắc nét.
5. **Trực quan hóa:** Xuất biểu đồ 4 bảng đối chiếu chuyên sâu.

In [ ]:
import os
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[✓] Thiết bị tính toán được chọn: {device}")

## PHẦN 1: ĐỊNH NGHĨA KIẾN TRÚC MÔ HÌNH VAE BẰNG PYTORCH

Encoder phân nhánh thành 2 đầu ra: vector kỳ vọng $\mu$ và vector log-phương sai $\log\sigma^2$.
Thủ thuật tái tham số hóa rút mẫu: $z = \mu + \sigma \odot \epsilon$ với $\epsilon \sim \mathcal{N}(0, I)$.

In [ ]:
from vae_pytorch import VariationalAutoencoder, DeterministicAutoencoder

vae = VariationalAutoencoder(input_dim=64, hidden_dim=128, latent_dim=2).to(device)
ae = DeterministicAutoencoder(input_dim=64, hidden_dim=128, latent_dim=2).to(device)
print("[✓] Đã khởi tạo VariationalAutoencoder và DeterministicAutoencoder.")

## PHẦN 2: HÀM MẤT MÁT CẬN DƯỚI BIẾN PHÂN (ELBO LOSS)

$$\mathcal{L}_{\text{ELBO}} = \text{BCE}(x, \hat{x}) + \beta \cdot D_{\text{KL}}\left(\mathcal{N}(\mu, \sigma^2) \parallel \mathcal{N}(0, I)\right)$$
$$D_{\text{KL}} = -\frac{1}{2} \sum_{j=1}^J \left( 1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2 \right)$$

In [ ]:
from vae_pytorch import elbo_loss_function
print("[✓] Đã sẵn sàng hàm elbo_loss_function.")

## PHẦN 3: BỘ SINH DỮ LIỆU TỔNG HỢP (OFFLINE DATASET)

In [ ]:
from vae_pytorch import generate_synthetic_digits_dataset

loader, tensor_x, tensor_y = generate_synthetic_digits_dataset(n_samples=600)
print(f"[*] Kích thước dữ liệu: {tensor_x.shape}, số lượng mẫu: {len(loader.dataset)}")

## PHẦN 4: ĐỐI CHIẾU GIÁ TRỊ HỌC THUẬT (ACADEMIC VALUE BENCHMARK)

Chạy thử nghiệm so sánh 3 trường phái:
1. **Mô hình 1: Deterministic Autoencoder (Hinton 2006)**
2. **Mô hình 2: VAE Không Điều Hòa KL ($\beta=0$)**
3. **Mô hình 3: Full VAE (Kingma & Welling 2013)**

In [ ]:
from vae_pytorch import run_academic_value_benchmark

# Chạy đối chuẩn giá trị học thuật
bench_results = run_academic_value_benchmark(device)

## PHẦN 5: TRỰC QUAN HÓA BẢN ĐỒ ĐA TẠP ẨN 2D (2D LATENT MANIFOLD GRID)

Quét lưới tọa độ 2D không gian ẩn $[-2.5, 2.5] \times [-2.5, 2.5]$ và giải mã để trực quan hóa sự biến đổi hình học mượt mà.

In [ ]:
from vae_pytorch import generate_manifold_grid

out_dir = "outputs"
generate_manifold_grid(bench_results["vae_model"], device, output_dir=out_dir, grid_size=12)

# Hiển thị ảnh bản đồ đa tạp
grid_img_path = os.path.join(out_dir, "vae_manifold_grid.png")
if os.path.exists(grid_img_path):
    img_grid = plt.imread(grid_img_path)
    plt.figure(figsize=(9, 9))
    plt.imshow(img_grid)
    plt.axis('off')
    plt.title("Bản Đồ Đa Tạp Không Gian Ẩn 2D Của VAE (Kingma & Welling 2013)", fontsize=13, fontweight='bold')
    plt.show()

## PHẦN 6: BIỂU ĐỒ ĐỐI CHIẾU HỌC THUẬT CHUYÊN SÂU

Xuất biểu đồ 4 bảng phân tích:
1. **Sự giằng co của ELBO:** Tái tạo (BCE) vs Điều hòa (KL).
2. **Cấu trúc không gian ẩn:** VAE được chuẩn hóa chặt chẽ về $\mathcal{N}(0, I)$.
3. **Chất lượng mẫu sinh ngẫu nhiên:** VAE sinh ảnh sắc nét so với ảnh rác của Autoencoder cũ.
4. **Mật độ bán kính:** Khẳng định các mẫu phân bố đều quanh tâm gốc tọa độ.

In [ ]:
from vae_pytorch import plot_comprehensive_academic_comparison

plot_comprehensive_academic_comparison(bench_results, output_dir=out_dir)

# Hiển thị biểu đồ đối chiếu
comp_img_path = os.path.join(out_dir, "vae_academic_comparison.png")
if os.path.exists(comp_img_path):
    img_comp = plt.imread(comp_img_path)
    plt.figure(figsize=(16, 12))
    plt.imshow(img_comp)
    plt.axis('off')
    plt.title("Kết Quả Đối Chiếu Học Thuật Toàn Diện (VAE ICLR 2014)", fontsize=14, fontweight='bold')
    plt.show()

## 🎓 KẾT LUẬN & Ý NGHĨA LỊCH SỬ CỦA VAE (ICLR 2014)

1. **Bước nhảy vọt sang AI Tạo Sinh:** Chuyển đổi mô hình nén dữ liệu tất định thành mô hình xác suất tạo sinh có khả năng sinh mẫu mới bất tận.
2. **Reparameterization Trick:** Đóng góp kỹ thuật kinh điển giúp nối liền cầu nối toán học giữa quá trình ngẫu nhiên và lan truyền ngược khả vi.
3. **Không gian ẩn liên tục, không lỗ hổng:** Nhờ thành phần phạt KL Divergence, không gian ẩn của VAE là một khối cầu Gauss mượt mà, cho phép nội suy liên tục giữa các khái niệm.
4. **Nền móng cho Diffusion Models:** Các mô hình khuếch tán tiên tiến nhất hiện nay (như Stable Diffusion) đều sử dụng VAE để nén không gian điểm ảnh khổng lồ xuống không gian ẩn trước khi thực hiện quá trình khuếch tán.